<a href="https://colab.research.google.com/github/iqraghori/Data-science/blob/main/Pandas_practice/merging_joining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

DATA DESCRIPTION
```
file name -> Columns
quater-i.csv -> ['order_id', 'quantity', 'item_id', 'choice_description_id' 'item_price']
items.csv -> ['item_id', 'item_name']
```
Dataset Link - https://drive.google.com/drive/folders/1Z0kaFybvgFeczeUj4dldUnhTdloLqLsL?usp=share_link

In [5]:
# import like this
import gdown
folder_url = 'https://drive.google.com/drive/folders/1Z0kaFybvgFeczeUj4dldUnhTdloLqLsL?usp=share_link'
gdown.download_folder(folder_url, quiet=False)

items_path = "data/items.csv"
q1_path = "data/quarter-1.csv"
q2_path = "data/quarter-2.csv"
q3_path = "data/quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

Retrieving folder contents


Processing file 1NpCPtn7zN24-tuSs-UYdaJjZrCiQV0qN IPL_Ball_by_Ball_2008_2022.csv
Processing file 1-kvv_9KCSAFWcrhS9WgTxSrURkRh6GNt ipl_deliveries.csv
Processing file 1w7DYDW13hfj99S3DGAng1CmTPOlQ5S3Y IPL_Matches_2008_2022.csv
Processing file 1MokLL6WAVRrA4TcWSCocTG_U7H-iN4Nw items.csv
Processing file 17Tca-FYOu-qMpAaJBUuC2E1Ex3jCMpFg quarter-1.csv
Processing file 1pIBukQRAJi5ksNR5lNj0qslIbrlS_Bzf quarter-2.csv
Processing file 19d3AsHDuTj5ZiH-HthdfLmEBWF4T-Oyd quarter-3.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NpCPtn7zN24-tuSs-UYdaJjZrCiQV0qN
To: /content/data/IPL_Ball_by_Ball_2008_2022.csv
100%|██████████| 19.9M/19.9M [00:00<00:00, 227MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-kvv_9KCSAFWcrhS9WgTxSrURkRh6GNt
To: /content/data/ipl_deliveries.csv
100%|██████████| 30.6M/30.6M [00:00<00:00, 195MB/s]
Downloading...
From: https://drive.google.com/uc?id=1w7DYDW13hfj99S3DGAng1CmTPOlQ5S3Y
To: /content/data/IPL_Matches_2008_2022.csv
100%|██████████| 471k/471k [00:00<00:00, 86.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MokLL6WAVRrA4TcWSCocTG_U7H-iN4Nw
To: /content/data/items.csv
100%|██████████| 1.08k/1.08k [00:00<00:00, 3.85MB/s]
Downloading...
From: https://drive.google.com/uc?id=17Tca-FYOu-qMpAaJBUuC2E1Ex3jCMpFg
To: /content/data/quarter-1.csv
100%|██████████| 44.4k/44.4k [00:00<00:00, 29.8MB/s]
Downloading...

###`Q:1-5`
1. You are given three quater files, your job is to append these three files and make a single dataframe.
2. Have a index as Q-1 Q-2 Q-3 for respective quater files in the dataframe
3. Your are given a file items.csv which has item_id and item_name. Find out most sold items in each quarter.
4. Find out items which has made most revenue in each quarter.
5. Find out avg order price of each quarter.

***Note: item_price is given as str with $ sign, in earlier task you have converted this to rupees, here too first convert item_price field in rupees.***

In [6]:
# code here
df_q = pd.concat([q1,q2,q3],keys=['Q_1','Q_2','Q_3'])
df_q.reset_index(level = 0, inplace = True)
df_q.rename(columns = {'level_0':'Quarter'}, inplace = True)
df = items.merge(df_q,how="inner",on="item_id")


In [7]:
df.head(2)

,item_id,item_name,Quarter,order_id,quantity,choice_description_id,item_price
0,0,Chips and Fresh Tomato Salsa,Q_1,13,1,0,$2.39
1,0,Chips and Fresh Tomato Salsa,Q_1,82,1,0,$2.95


In [8]:
# Most sold items
df.groupby(['Quarter','item_name'])['quantity'].sum().reset_index().sort_values(['quantity'],ascending=False).groupby('Quarter').first()

,item_name,quantity
Quarter,,
Q_1,Chicken Bowl,367
Q_2,Chicken Bowl,394


In [9]:
df['item_price'] = df['item_price'].str.strip().str.replace('$','').astype(float)

In [10]:
# products made most revenue in each qaurter
#revenue

df['revenue'] = df['item_price']* df['quantity']
print("highest Revenue Items per Quarter:")
df.groupby(['Quarter','item_name'])['revenue'].sum().reset_index().sort_values(['revenue'],ascending=False).groupby('Quarter').first()

highest Revenue Items per Quarter:


,item_name,revenue
Quarter,,
Q_1,Chicken Bowl,3852.38
Q_2,Chicken Bowl,4192.25


In [11]:
#avg order price of each quarter
order_totals = df.groupby(['Quarter','order_id'])['revenue'].sum().reset_index()

avg_order_price = (
    order_totals
    .groupby('Quarter')['revenue']
    .mean()
    .reset_index()
)
print("Average Order Price per Quarter (in PKR):")
print(avg_order_price)



Average Order Price per Quarter (in PKR):
  Quarter    revenue
0     Q_1  13.809488
1     Q_2  13.279828


###`Q-6` From the IPL wala dataset you have to find the Purple cap holder each season.

*Note: Bowler with most no wickets in a season gets purple cap. If more than one bowler have same no of wickets in the season, one with least ecomnomy among them is purple cap holder.*

Bowler's Economy = runs-conceded per six balls

In [12]:
# code here
ipl_ball_by_ball = pd.read_csv("data/IPL_Ball_by_Ball_2008_2022.csv")
ipl_match = pd.read_csv("data/IPL_Matches_2008_2022.csv")
ipl_deliveries = pd.read_csv("data/ipl_deliveries.csv")

In [20]:
df_ipl = ipl_deliveries.merge(ipl_match, how='inner', on='ID')

# --- Wickets ---
bowler_credit = ['caught', 'bowled', 'lbw', 'caught and bowled', 'stumped', 'hit wicket']
df_wickets = df_ipl[df_ipl['kind'].isin(bowler_credit)]

wickets = (
    df_wickets
    .groupby(['Season', 'bowler'])['kind']
    .count()
    .reset_index()
    .rename(columns={'kind': 'wickets'})
)

# --- Economy ---
df_ipl['bowlRuns'] = df_ipl.apply(
    lambda x: x['batsman_run'] if x['extra_type'] not in ['legbyes', 'byes']  # ✅ Bug 1 & 2 fixed
    else x['total_run'], axis=1
)
df_ipl['is_legal'] = ~df_ipl['extra_type'].isin(['wides', 'noballs'])

economy = (
    df_ipl
    .groupby(['Season', 'bowler'])
    .agg(
        runs_conceded=('bowlRuns', 'sum'),
        balls=('is_legal', 'sum')           # ✅ Bug 3 fixed — sum not count
    )
    .reset_index()
)
economy['economy'] = (economy['runs_conceded'] / economy['balls'] * 6).round(2)

# --- Combine & Pick Purple Cap ---
df_bowler = wickets.merge(economy[['Season', 'bowler', 'economy']], on=['Season', 'bowler'])

purple_cap = (
    df_bowler
    .sort_values(['Season', 'wickets', 'economy'], ascending=[True, False, True])  # ✅ Bug 4 fixed
    .groupby('Season')
    .first()
    .reset_index()
)

print(purple_cap[['Season', 'bowler', 'wickets', 'economy']])

     Season         bowler  wickets  economy
0   2007/08  Sohail Tanvir       22     6.19
1      2009       RP Singh       23     6.80
2   2009/10        PP Ojha       21     7.27
3      2011     SL Malinga       28     5.89
4      2012       M Morkel       25     7.16
5      2013       DJ Bravo       32     7.81
6      2014      MM Sharma       23     8.47
7      2015       DJ Bravo       26     8.25
8      2016        B Kumar       23     7.26
9      2017        B Kumar       26     7.13
10     2018         AJ Tye       24     7.89
11     2019    Imran Tahir       26     6.76
12  2020/21       K Rabada       32     8.26
13     2021       HV Patel       32     7.78
14     2022      YS Chahal       27     7.57


###`Q-7:` Best bowler in death overs.
*Note: Have taken most no of wickets in case of tie with least economy*

Death Overs - [16-20]

In [ ]:
# code here

###`Q-8` Batsman record season wise

Make a function which takes a input `batsman_name` and it returns a dataframe.
Columns of the data frame are - `['Season','Innings', 'TotalRuns', 'Avg', 'HighestScore','StrikeRate']`.
* In result make `Season` column as index.

* Avg - total_runs/ no of time got out. - player_out column will help.
* StrikeRate -(total_runs/ balls faced) * 100- wides are not included in batsman ball faced counts. No balls are included. -> Extra_type column will help
* Batsman Can score runs on No Balls.
* Batsman can get out on No Ball or Wides. And even while being on non-striker. Keep these things in mind before masking.

In [ ]:
# code here

###`Q-9` Using both dataset, make a dataframe as described below

Data Frame columns-> `['PlayerOfThematch', 'BattingFigure', 'BowlingFigure']`

* BattingFigure->`<runs>/<balls>`
* BowlingFigure->`<wicket>/<runs-conceded>`

DataFrame should have one record for each match.

Say 'V Kohli' got POM award then in dataset include his batting figure of that match. Say he scored 112runs in 76 balls. And he hasn't bowled so Bowling Figure will be NaN
```
PlayerOfThematch BattingFigure BowlingFigure
V Kohli          112/76         nan  

```


In [ ]:
# code here

## **Questions Based on Iris Dataset**

- **Sepal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **SepalLengthCm:** Sepal length of flowers in cm
    - **SepalWidthCm:** Sepal width of flowers in cm

- **Petal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **PetalLengthCm:** Petal length of flowers in cm
    - **PetalWidthCm:** Petal width of flowers in cm

- **Iris Virginica:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Versicolor:** https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Setosa:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv
    - **Unnamed 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

In [ ]:
import pandas as pd
sepal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv")
petal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv")

virginica = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv")
versicolor = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv")
setosa = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv")


### `Q-9:` Plot a bar chart of the average Sepal Length  of Virginica and average Petal length of Setosa flower.

In [ ]:
# code here

### `Q-10:` Create the complete dataset by uisng the below datasets:
- virginica
- versicolor
- setosa
- sepal all
- petal all

This dataset should have these below column names in order:
1. Id
2. Species
3. SepalLengthCm
4. SepalWidthCm
5. PetalLengthCm
6. PetalWidthCm

Also, the dataset should be shuffled means the `Id` column should not be in increasing or decreasing order. So, make a dataset which has the shuffled Id column. You can use `DataFrame.sample()` method to shuffle.

In [ ]:
# code here

### `Q-11:` Find out the maximum and minimum sepal width and petal width of Setosa and Versicolor. To do this:
- First create a dataset with merging the required datasets
- After that, use `groupby` to create groups based on the "Species" column.
- Then find out which are asked in this question.


The output should be like this:
```bash
Minimum Sepal width of Setosa is 2.3
Maximum Sepal width of Setosa is 4.4

**************************************************

Minimum Sepal width of Versicolor is 2.0
Maximum Sepal width of Versicolor is 3.4

**************************************************
```

In [ ]:
# code here